<div class='bar_title'></div>

*Simulation for Decision Making (S4DM)*

# Simulation Modeling Using SimPy (Part 3)

Gunther Gust & Govind Rao <br>
Chair for Enterprise AI <br>
Data Driven Decisions Group <br>
Center for Artificial Intelligence and Data Science (CAIDAS)

<img src="images/d3.png" alt="D3 Group logo" style="width:20%; float:left;" />


<img src="images/CAIDASlogo.png" alt="CAIDAS logo" style="width:20%; float:left;" />


# Today: Still at Model Translation

We are still in the **Model Translation** step of the simulation study — turning our conceptual model into a running computer model. Today we extend our SimPy toolbox so we can translate more realistic systems.

<img src="images/simulation_study_steps_translation.png" alt="Seven-step simulation study, highlighting the Model Translation step" style="width:100%; float:left;" />


# Recap: What we can model so far

| Concept | Example |
|---|---|
| **Process** — a Python generator describing an entity's behavior over time, yielding events (e.g., `env.timeout`) to advance the simulation clock. | In **park-and-drive**, the car's process alternates between driving and parking, each step yielding a timeout for its duration. |
| **Process interaction** — one process waits for another to finish via `yield env.process(...)`. | In the **car wash**, a `Car` process starts and then `yield`s on the washing-machine process, resuming only once the wash is done. |
| **Shared resources** — a limited number of identical slots that entities request and release, queueing up when the resource is busy. | In the **Mensa**, students share the single cashier: each student requests the resource, pays, and releases it so the next in line can go. |

* **Modeling conventions** — we consistently model resources as Python classes, entities as Python classes with a `run()` method, and entity arrivals as a generator function that we register as a SimPy process via `env.process(...)`.


# Agenda for today

We extend the SimPy toolbox in three directions, each motivated by a real-world example:

1. **Condition events** — *what if a process needs to wait for more than one thing at the same time?*
    * *Example:* A **bank customer** runs out of patience and leaves the queue.
2. **Advanced resources** — *what if resources have priorities, preemption, or bulk-good semantics?*
    * *Example 1:* A **critical patient** needs priority access to the only doctor.
    * *Example 2:* Three **cars** share a finite 200-liter fuel tank at a gas station.
3. **Process interruptions** — *what if one process needs to cancel another?*
    * *Example:* A **driver** returns before her electric car has finished charging.

Credits: The following content is adapted from the official [SimPy documentation](https://simpy.readthedocs.io/en/latest/).


# Part A: Condition Events

## Motivation

So far, a process can only wait for **one** event at a time — a timeout, a resource request, or another process.

But real entities often wait for **several things simultaneously**, and the flow depends on **which events happen, and in which combination**:

* A **bank customer** waits for the counter — **or** runs out of patience and leaves
* A **car** at a traffic light waits for the green light — **or** until an emergency vehicle clears the intersection
* A **waiter** serves a table only once **both** the food **and** the drinks are ready

With our current toolbox, none of these cases can be modeled cleanly. **Condition events** close this gap.


## Example: Impatient Bank Customers

### Scenario

* **Customers** randomly arrive at a bank and want to be served (e.g., to withdraw money).
* The bank has **one counter** with a random service time.
* The customers form **a queue** in front of the counter.
* Each customer has a certain **willingness to wait**. If their patience runs out, they leave the queue **without being served**.


In [1]:
# Imports used throughout the notebook.
import simpy
import random

from simpy.events import AnyOf, AllOf, Event


### Resource (bank counter):

> *Modeling pattern:* every entity/resource class stores `self.env` in `__init__` so its methods can schedule timeouts and start child processes.


In [2]:
# Resource class: single bank counter with a random service time.
class BankCounter:

    def __init__(self, env, serving_time):
        self.env = env
        self.counter = simpy.Resource(env, capacity=1)
        self.serving_time = serving_time

    def serve(self, customer):
        duration = random.expovariate(1.0 / self.serving_time)  # random serving time
        yield self.env.timeout(duration)


### Entity (customer):

In [3]:
# Entity: customer either gets served at the counter OR leaves when patience runs out.
class Customer:
    def __init__(self, env, name):
        self.env = env
        self.name = name

    def run(self, bank_counter):
        arrival_time = self.env.now
        print(f't={arrival_time:5.1f} {self.name} arrives')

        with bank_counter.counter.request() as req:
            patience = random.uniform(MIN_PATIENCE, MAX_PATIENCE)  # random patience

            # CONDITION EVENT: Wait for the counter OR abort due to impatience
            results = yield req | self.env.timeout(patience)

            wait = self.env.now - arrival_time

            # Depending on the CONDITION EVENT, split the flow into A) and B)
            if req in results:  # A) The customer got to the counter
                print(f't={self.env.now:5.1f} {self.name} waited {wait:.1f}')
                yield self.env.process(bank_counter.serve(self.name))
                print(f't={self.env.now:5.1f} {self.name} finished')

            else:  # B) The customer left due to impatience
                print(f't={self.env.now:5.1f} {self.name} LEFT after {wait:.1f}')


**What's new here?**

* Line `results = yield req | self.env.timeout(patience)` is a **condition event**: the customer waits for *whichever happens first* — getting access to the counter (`req`) **or** running out of patience.
  * The `|` is the shorthand for `AnyOf` — we formalize the syntax in the next section.
* The return value `results` is a dictionary containing only the event(s) that actually triggered, so we check `if req in results` to decide whether the customer was served or left.
* The entity flow **splits** into two branches (A: served, B: left) depending on that outcome — something we could not do cleanly with the earlier toolbox.


### Entity Generation

In [4]:
# Arrival process: generate customers with exponentially-distributed inter-arrival times.
def customer_generator(env, num_customers, mean_interval, bank_counter):
    """Create `num_customers` customers with exponentially-distributed inter-arrival times."""
    for i in range(num_customers):
        customer = Customer(env, f'Customer {i+1}')  
        env.process(customer.run(bank_counter))       # start the customer process

        t = random.expovariate(1.0 / mean_interval)   # wait a random time before creating the next one
        yield env.timeout(t)


### Run the simulation

When you run this, watch for **both outcomes** — some customers get served, others leave when their patience expires.


In [5]:
# Run: 10 customers arriving over ~100 minutes; some are served, some leave.
RANDOM_SEED = 42
NUM_CUSTOMERS = 10        # Total number of customers
MEAN_INTERARRIVAL = 10    # New customers roughly every 10 minutes
MIN_PATIENCE = 1          # Min. customer patience (minutes)
MAX_PATIENCE = 3          # Max. customer patience (minutes)
SERVING_TIME = 12.0       # Avg. serving time (minutes)

random.seed(RANDOM_SEED)

env = simpy.Environment()
bank_counter = BankCounter(env, SERVING_TIME)
env.process(customer_generator(env, NUM_CUSTOMERS, MEAN_INTERARRIVAL, bank_counter))
env.run()


t=  0.0 Customer 1 arrives
t=  0.0 Customer 1 waited 0.0
t=  3.9 Customer 1 finished
t= 10.2 Customer 2 arrives
t= 10.2 Customer 2 waited 0.0
t= 12.7 Customer 3 arrives
t= 13.9 Customer 3 LEFT after 1.2
t= 23.8 Customer 2 finished
t= 35.0 Customer 4 arrives
t= 35.0 Customer 4 waited 0.0
t= 38.0 Customer 4 finished
t= 40.5 Customer 5 arrives
t= 40.5 Customer 5 waited 0.0
t= 43.1 Customer 5 finished
t= 47.5 Customer 6 arrives
t= 47.5 Customer 6 waited 0.0
t= 50.5 Customer 6 finished
t= 58.0 Customer 7 arrives
t= 58.0 Customer 7 waited 0.0
t= 58.1 Customer 7 finished
t= 66.9 Customer 8 arrives
t= 66.9 Customer 8 waited 0.0
t= 71.9 Customer 8 finished
t= 83.3 Customer 9 arrives
t= 83.3 Customer 9 waited 0.0
t= 85.0 Customer 10 arrives
t= 86.2 Customer 10 LEFT after 1.2
t= 88.2 Customer 9 finished


**What to look at:** Some customers print `Finished` (they got the counter in time), others `LEFT` (their patience ran out while waiting). Scan the output for both outcomes — that split is exactly what the condition event `req | env.timeout(patience)` enables.


## Syntax of Condition Events (reference)

You've just used `|` in the bank example. Here is the formal reference behind it.


SimPy offers `AnyOf` and `AllOf` events, which are both *condition events*.

```python
from simpy.events import AnyOf, AllOf, Event

events = [Event(env) for i in range(3)]
a = AnyOf(env, events)  # Triggers if AT LEAST ONE of "events" is triggered.
b = AllOf(env, events)  # Triggers if ALL of "events" are triggered.
```

* `AnyOf` fires as soon as **at least one** event in the list triggers.
* `AllOf` waits until **all** events have triggered.

As a **shorthand**, you can also use the logical operators:

* `|` (or) — equivalent to `AnyOf`
* `&` (and) — equivalent to `AllOf`


### Return Value

* When a condition event fires, its **return value** is an ordered dictionary with an entry for **every triggered event**.
* The event instances are used as **keys**, and the **event values** will be the **values**.
  * Example: after `ret = yield t1 | t2` (with `t1` firing first at value `'spam'`), `ret == {t1: 'spam'}`.
* For `AllOf`, the size of that dictionary equals the length of the event list.
* For `AnyOf`, the dictionary has **at least one** entry.

Let's have a precise look a this in the following example:


**Demo:** Two timeouts fire at t=1 and t=2. `t1` carries the value `'spam'`, `t2` the value `'eggs'`. `AnyOf` returns after the **first** one triggers (t=1), so the result dict has a single entry: `t1` mapped to its value `'spam'`.


In [6]:
# Syntax demo: AnyOf fires as soon as the FIRST event in the list is triggered.
def demo_anyof_list(env):
    t1, t2 = env.timeout(1, value='spam'), env.timeout(2, value='eggs')  # two events
    events = [t1, t2]                                                     # put in a list
    ret = yield AnyOf(env, events)                                        # wait until one fires
    print(ret)

env = simpy.Environment()
env.process(demo_anyof_list(env))
env.run()


<ConditionValue {<Timeout(1, value=spam) object at 0x108b5f500>: 'spam'}>


### Custom Events (manual triggering)

So far we've yielded on things SimPy fires automatically — `env.timeout(n)` (clock), `resource.request()` (scheduler), `env.process(gen)` (child finishes). **A `simpy.Event` is a bare event *you* fire by hand.** It stays inactive until some piece of code calls `.succeed()` (or `.fail()`), at which point every process waiting on it wakes up.

| Yield statement | Who fires it? |
|---|---|
| `yield env.timeout(n)` | SimPy's clock, automatically after `n` |
| `yield resource.request()` | SimPy's resource scheduler, when a slot frees |
| `yield env.process(process)` | SimPy, when the child process finishes |
| **`yield signal`** (with `signal = simpy.Event(env)`) | **You, explicitly, via `.succeed()` or `.fail()`** |

```python
signal = simpy.Event(env)         # bare event, not yet triggered
signal.succeed(value='go!')       # fire it (optionally with a return value)
signal.fail(RuntimeError('x'))    # or fire it as a failure (throws into yielder)
```

Typical uses: a doorbell any process can ring, a guard that flips a flag once a condition holds, a cancellation signal one process raises to release others.

Because `Event` is the same object type condition events operate on, you can mix it into `|` / `&`:

```python
yield signal | env.timeout(60)    # wake on manual signal OR after 60 min timeout
```


In [7]:
# Custom event: 'waiter' blocks on a bare Event until 'trigger' fires it by hand.
def waiter(env, signal):
    print(f't={env.now:5.1f} waiter: waiting for the signal...')
    value = yield signal                       # blocks until someone fires `signal`
    print(f't={env.now:5.1f} waiter: woke up, got value={value!r}')

def trigger(env, signal):
    yield env.timeout(5)                       # simulate "something happens" after 5 min
    print(f't={env.now:5.1f} trigger: firing the signal!')
    signal.succeed(value='go!')                # <-- manual trigger

env = simpy.Environment()
signal = simpy.Event(env)                      # bare event, not yet triggered

env.process(waiter(env, signal))
env.process(trigger(env, signal))

env.run()


t=  0.0 waiter: waiting for the signal...
t=  5.0 trigger: firing the signal!
t=  5.0 waiter: woke up, got value='go!'


**Business examples.** Two common patterns where this primitive is exactly right:

1. **Payment-confirmed → ship** *(one waiter).* The order-fulfillment process yields on a `payment_ok` event; the payment gateway fires `payment_ok.succeed()` when the bank confirms. Arrival time is unpredictable — not a timeout, not a resource queue.
2. **Flash sale starts at 12:00** *(fan-out, many waiters).* Thousands of customer processes all `yield sale_starts`. **One** `.succeed()` call at t=12:00 wakes **all** of them in the same sim-time instant — the "one fires, many wake" pattern is where `Event` really differs from timeouts.


---

**Where are we?** Processes can now wait for *multiple* events with `|`/`&` — and for **custom signals** that other processes fire by hand. Next: richer resource types — priorities, preemption, bulk goods, distinguishable objects.


# Part B: Advanced Resources

## Resource types in SimPy

A shared resource is a **congestion point**: processes queue up to use it.

SimPy offers different resource types for different needs. **We'll see each of these in turn in the rest of Part B:**

| Type | What it models | Example |
|---|---|---|
| `Resource` | Limited number of slots | Bank counter, gas pumps |
| `PriorityResource` | Slots with request priorities | ER triage, VIP queue |
| `PreemptiveResource` | High-priority requests kick low-priority out | Machine repair vs. other jobs |
| `Container` | Bulk goods (continuous or discrete) | Fuel tank, water reservoir |
| `Store` | Distinguishable Python objects | Spare-parts warehouse, restaurant orders |


## Inspecting a Resource

Let's **revisit the bank example** from Part A — but this time with **patient customers** who simply wait for the counter instead of giving up. This gives us a clean view of how a basic `simpy.Resource` behaves over time.

A resource exposes several attributes you can read at any simulation time:

* `res.capacity` — total number of slots
* `res.count`    — how many slots are currently in use
* `res.users`    — list of request events currently holding a slot
* `res.queue`    — list of request events still waiting

Let's watch how these change as the patient customers move through the bank:

In [8]:
# Inspecting a Resource: watch users and queue as patient customers pass through the bank.
NUM_CUSTOMERS = 4

class Bank:
    """A bank with `num_counters` parallel counters."""
    def __init__(self, env, num_counters, serving_time):
        self.env = env
        self.counter = simpy.Resource(env, capacity=num_counters)
        self.serving_time = serving_time

    def serve(self, customer):
        yield self.env.timeout(self.serving_time)

def print_stats(bank, label):
    c = bank.counter
    users_names  = [r.name for r in c.users]                  # names currently served
    queued_names = [r.name for r in c.queue]                  # names waiting in queue
    print(f't={bank.env.now:5.1f}  {label:22s}  '
          f'{c.count}/{c.capacity} counters busy | '
          f'users={users_names}, queued={queued_names}')

class PatientCustomer:
    def __init__(self, env, name):
        self.env = env
        self.name = name

    def run(self, bank):
        print_stats(bank, f'{self.name} arrives')
        with bank.counter.request() as req:
            # SimPy request objects are plain Python objects; we attach our own
            # `.name` attribute so the trace shows readable names.
            req.name = self.name
            yield req
            print_stats(bank, f'{self.name} served')
            yield self.env.process(bank.serve(self.name))
            print_stats(bank, f'{self.name} leaves')

env = simpy.Environment()
bank = Bank(env, num_counters=1, serving_time=5)
for i in range(1, NUM_CUSTOMERS + 1):
    env.process(PatientCustomer(env, f'Customer {i}').run(bank))
env.run()


t=  0.0  Customer 1 arrives      0/1 counters busy | users=[], queued=[]
t=  0.0  Customer 2 arrives      1/1 counters busy | users=['Customer 1'], queued=[]
t=  0.0  Customer 3 arrives      1/1 counters busy | users=['Customer 1'], queued=['Customer 2']
t=  0.0  Customer 4 arrives      1/1 counters busy | users=['Customer 1'], queued=['Customer 2', 'Customer 3']
t=  0.0  Customer 1 served       1/1 counters busy | users=['Customer 1'], queued=['Customer 2', 'Customer 3', 'Customer 4']
t=  5.0  Customer 1 leaves       1/1 counters busy | users=['Customer 1'], queued=['Customer 2', 'Customer 3', 'Customer 4']
t=  5.0  Customer 2 served       1/1 counters busy | users=['Customer 2'], queued=['Customer 3', 'Customer 4']
t= 10.0  Customer 2 leaves       1/1 counters busy | users=['Customer 2'], queued=['Customer 3', 'Customer 4']
t= 10.0  Customer 3 served       1/1 counters busy | users=['Customer 3'], queued=['Customer 4']
t= 15.0  Customer 3 leaves       1/1 counters busy | users=['Cust

### ❓ Menti: `users` & `queue` attributes


We've now seen how the basic `Resource` behaves. Next: what if some requesters should jump the queue?


## PriorityResource — Emergency Room

In real systems, not all requests are equally important.
A critical patient cannot wait behind routine check-ups.

SimPy's `PriorityResource` lets each `request()` call carry a priority value:

```python
with resource.request(priority=1) as req:   # smaller number = higher priority
    yield req
    ...
```

* **Smaller number = higher priority** — higher-priority requests are served before lower-priority ones.


### Scenario: Emergency Room with 1 doctor

* Treatment takes **10 minutes** per patient.
* Three patients arrive at t=0, 1, 2 (see code below) — two routine, one critical.


In [9]:
# PriorityResource: critical patient (priority 0) jumps the routine queue (priority 2).
class EmergencyRoom:
    """An emergency room with one doctor (a PriorityResource)."""
    def __init__(self, env, num_doctors, treatment_time):
        self.env = env
        self.doctor = simpy.PriorityResource(env, capacity=num_doctors)   # NEW: PriorityResource instead of Resource
        self.treatment_time = treatment_time

    def treat(self, patient_name):
        yield self.env.timeout(self.treatment_time)

class PriorityPatient:
    def __init__(self, env, name, priority, arrival):
        self.env = env
        self.name = name
        self.priority = priority                                          # NEW: each patient carries a priority
        self.arrival = arrival

    def run(self, er):
        yield self.env.timeout(self.arrival)                              # wait until arrival time
        with er.doctor.request(priority=self.priority) as req:            # NEW: priority passed to request()
            print(f't={self.env.now:5.1f} {self.name} requesting (priority={self.priority})')
            yield req
            print(f't={self.env.now:5.1f} {self.name} is being treated')
            yield self.env.process(er.treat(self.name))
            print(f't={self.env.now:5.1f} {self.name} leaves')

env = simpy.Environment()
er = EmergencyRoom(env, num_doctors=1, treatment_time=10)

# Create three patients directly (lower number = higher priority)
env.process(PriorityPatient(env, 'Patient 1 (routine)',  priority=2, arrival=0).run(er))
env.process(PriorityPatient(env, 'Patient 2 (routine)',  priority=2, arrival=1).run(er))
env.process(PriorityPatient(env, 'Patient 3 (CRITICAL)', priority=0, arrival=2).run(er))   # will jump the queue


<Process(run) object at 0x108b7ca70>

### ❓ Menti: Order of treatment (priority only)


In [10]:
env.run()

t=  0.0 Patient 1 (routine) requesting (priority=2)
t=  0.0 Patient 1 (routine) is being treated
t=  1.0 Patient 2 (routine) requesting (priority=2)
t=  2.0 Patient 3 (CRITICAL) requesting (priority=0)
t= 10.0 Patient 1 (routine) leaves
t= 10.0 Patient 3 (CRITICAL) is being treated
t= 20.0 Patient 3 (CRITICAL) leaves
t= 20.0 Patient 2 (routine) is being treated
t= 30.0 Patient 2 (routine) leaves


## PreemptiveResource

Sometimes, new requests are **so important** that queue-jumping is not enough — they need to **kick the current user off** the resource.

This is called **preemption**. `PreemptiveResource` allows exactly this behavior.

### Background: exception handling in Python

Recall: `try / except` catches exceptions. Here we use it to catch `simpy.Interrupt` when a process is preempted. See [W3Schools — Python try/except](https://www.w3schools.com/python/python_try_except.asp) if you need a refresher.

<img src="images/try_except.png" alt="Python try/except syntax diagram" style="width:50%;" />


### ER revisited: the critical patient preempts

Same scenario, but now Patient 3 is so urgent that treatment of Patient 1 is interrupted:

In [11]:
# PreemptiveResource: critical patient interrupts the patient currently being treated.
class PreemptiveER:
    """An emergency room with one doctor (a PreemptiveResource)."""
    def __init__(self, env, num_doctors, treatment_time):
        self.env = env
        self.doctor = simpy.PreemptiveResource(env, capacity=num_doctors)  # NEW: PreemptiveResource
        self.treatment_time = treatment_time

    def treat(self, patient_name):
        yield self.env.timeout(self.treatment_time)

class PreemptivePatient:
    def __init__(self, env, name, priority, arrival):
        self.env = env
        self.name = name
        self.priority = priority
        self.arrival = arrival

    def run(self, er):
        yield self.env.timeout(self.arrival)
        with er.doctor.request(priority=self.priority) as req:
            print(f't={self.env.now:5.1f} {self.name} requesting (priority={self.priority})')
            yield req
            print(f't={self.env.now:5.1f} {self.name} is being treated')

            try:                                                           # treatment may be preempted
                yield self.env.process(er.treat(self.name))
                print(f't={self.env.now:5.1f} {self.name} leaves')
            except simpy.Interrupt as interrupt:
                # interrupt.cause is a SimPy `Preempted` object with `.by` (the preempting
                # request) and `.usage_since` (when this request started using the resource).
                preempted_by = interrupt.cause.by
                time_treated = self.env.now - interrupt.cause.usage_since
                print(f't={self.env.now:5.1f} {self.name} was PREEMPTED by {preempted_by} '
                      f'after {time_treated:.1f} min of treatment')

env = simpy.Environment()
er = PreemptiveER(env, num_doctors=1, treatment_time=10)

env.process(PreemptivePatient(env, 'Patient 1 (routine)',  priority=2, arrival=0).run(er))
env.process(PreemptivePatient(env, 'Patient 2 (routine)',  priority=2, arrival=1).run(er))
env.process(PreemptivePatient(env, 'Patient 3 (CRITICAL)', priority=0, arrival=2).run(er))


<Process(run) object at 0x108b7dc10>

### ❓ Menti: Fate of Patient 1 under preemption


In [12]:
env.run()

t=  0.0 Patient 1 (routine) requesting (priority=2)
t=  0.0 Patient 1 (routine) is being treated
t=  1.0 Patient 2 (routine) requesting (priority=2)
t=  2.0 Patient 3 (CRITICAL) requesting (priority=0)
t=  2.0 Patient 1 (routine) was PREEMPTED by <Process(run) object at 0x108b7dc10> after 2.0 min of treatment
t=  2.0 Patient 3 (CRITICAL) is being treated
t= 12.0 Patient 3 (CRITICAL) leaves
t= 12.0 Patient 2 (routine) is being treated
t= 22.0 Patient 2 (routine) leaves


**Note:** `PreemptiveResource` values **priorities higher than preemption**. Preempt requests cannot bypass a higher-prioritized request.

*Example:* if a priority-0 request is already queued, an incoming preempt request with priority 1 will **not** preempt — the priority-0 request still gets access first.


## Container

So far, resources modeled **slots** (e.g., one counter, one doctor).

But what about **bulk goods** — fuel, water, apples, electricity?

`Container` models the production and consumption of **a bulk quantity with no individual items**.
It may be continuous (like water) or discrete (like apples).

> *Isn't that just a `Resource` with `capacity=200`?* No — `Resource` treats slots as integer check-outs of one each; `Container` lets processes `put()` or `get()` **arbitrary amounts** (e.g., 30 liters, 47.5 kg) in a single call.

**Gas station example:**
* Tankers **increase** the amount of gasoline in the tank
* Cars **decrease** it


### Syntax

```python
tank = simpy.Container(env, init=100, capacity=200)
yield tank.put(amount=50)   # add 50 units
yield tank.get(amount=20)   # take 20 units
```

### Mini gas station

In [13]:
# Container: three cars draw fuel from a shared 200-liter tank (bulk good).
PUMP_RATE = 10  # liters per time unit (minute)

class GasStation:
    def __init__(self, env, num_pumps, tank_init, tank_capacity):
        self.env = env
        self.pump = simpy.Resource(env, capacity=num_pumps)
        self.tank = simpy.Container(env, init=tank_init, capacity=tank_capacity)

    def refuel(self, amount):
        # Withdraw the bulk amount atomically, then simulate pumping time.
        yield self.tank.get(amount)
        yield self.env.timeout(amount / PUMP_RATE)

class Car:
    def __init__(self, env, name, fuel_needed):
        self.env = env
        self.name = name
        self.fuel_needed = fuel_needed

    def run(self, station):
        print(f't={self.env.now:5.1f} {self.name} arrives')
        with station.pump.request() as req:
            yield req
            print(f't={self.env.now:5.1f} {self.name} starts refueling '
                  f'(tank level before: {station.tank.level})')
            yield self.env.process(station.refuel(self.fuel_needed))
            print(f't={self.env.now:5.1f} {self.name} leaves '
                  f'(tank level after: {station.tank.level})')

env = simpy.Environment()
station = GasStation(env, num_pumps=1, tank_init=100, tank_capacity=200)

env.process(Car(env, 'Car 1', fuel_needed=30).run(station))
env.process(Car(env, 'Car 2', fuel_needed=40).run(station))
env.process(Car(env, 'Car 3', fuel_needed=25).run(station))

env.run()


t=  0.0 Car 1 arrives
t=  0.0 Car 2 arrives
t=  0.0 Car 3 arrives
t=  0.0 Car 1 starts refueling (tank level before: 100)
t=  3.0 Car 1 leaves (tank level after: 70)
t=  3.0 Car 2 starts refueling (tank level before: 70)
t=  7.0 Car 2 leaves (tank level after: 30)
t=  7.0 Car 3 starts refueling (tank level before: 30)
t=  9.5 Car 3 leaves (tank level after: 5)


## Store


So far we have seen two ways to model resources:

* **`Resource`** — interchangeable slots (any cashier is as good as any other)
* **`Container`** — bulk quantities (10 liters is 10 liters, regardless of which liters)

But what if the items themselves are **distinguishable** — each one a full Python object with its own attributes?

`simpy.Store` fills this gap. Processes `put()` and `get()` **Python objects** (anything: dicts, custom classes, tuples), and the objects keep their identity and attributes throughout.

```python
store = simpy.Store(env, capacity=10)
yield store.put(order)      # place an object into the store
item = yield store.get()    # retrieve the next object
```


### Application examples

**Spare-parts warehouse.** A workshop's warehouse holds a mix of distinct items (e.g., *"brake pad, model X, serial 1234"*, *"oil filter, model Y, serial 5678"*). Mechanics request specific part types, not just "one part." A `Store` lets you `put()` and `get()` these part objects as-is, so each mechanic can pick the part they actually need.

**Restaurant kitchen.** Waiters submit **orders** (each with a table number, a list of dishes, and a priority) into a kitchen queue. Cooks pull the next order, which is a full Python object with all its details — not an anonymous slot. A `Store` models this order-by-order flow cleanly.

Stores are **not covered further** in this course. See the [SimPy docs](https://simpy.readthedocs.io/en/latest/topical_guides/resources.html#stores) if you need them.

### Part B recap: resource zoo

| Type | Distinguishing feature | Example in this lecture |
|---|---|---|
| `Resource` | Interchangeable slots | Bank counter |
| `PriorityResource` | Priority per request (smaller = higher) | Emergency room — critical jumps the queue |
| `PreemptiveResource` | Priority + preemption of the current user | Emergency room — critical interrupts treatment |
| `Container` | Bulk amounts (`put(n)`, `get(n)`) | Gas station — shared 200-liter tank |
| `Store` | Distinguishable Python objects (`put(obj)`, `get()`) | Spare-parts warehouse *(mentioned only)* |


---

**Where are we?** Resources now handle priorities, preemption, bulk goods, and distinguishable items. Next: directly interrupting a running process — no resource needed.


# Part C: Process Interruptions

## Motivation

`PreemptiveResource` kicks a process off a **shared resource**.

But sometimes we need to interrupt a process directly — no resource involved:

* Stop charging an electric vehicle when an important trip comes up
* A phone call interrupts a cashier serving a customer
* A machine breaks down and the production process halts

## PreemptiveResource vs. `process.interrupt()`

| | `PreemptiveResource` | `process.interrupt()` |
|---|---|---|
| Who triggers? | A new high-priority request (via SimPy scheduler) | Another process, with an explicit call |
| Involves a resource? | Yes | No |
| Typical outcome | Victim re-queues, resumes later | Victim aborts |
| Example | Repair preempts other job | Driver cancels charging |

Both use `simpy.Interrupt` and `try / except` on the interrupted side.


## Syntax

```python
process = env.process(some_process(env))
...
process.interrupt('reason')   # throws simpy.Interrupt into the target process
```

## Example: Electric Vehicle Charging

### Scenario

* A driver searches a parking spot and charges the electric vehicle.
* If the driver returns **before** the charging has finished, the charging is interrupted.

In [14]:
# process.interrupt(): driver returns before charging finishes and cancels it directly.
class ChargingStation:
    def __init__(self, env, full_charge_time):
        self.env = env
        self.full_charge_time = full_charge_time

    def charge(self, ev_name):
        print(f't={self.env.now:5.1f} {ev_name} charging started')
        try:
            yield self.env.timeout(self.full_charge_time)
            print(f't={self.env.now:5.1f} {ev_name} charging done')
        except simpy.Interrupt as i:
            print(f't={self.env.now:5.1f} {ev_name} charging interrupted, msg: {i.cause}')

    def abort_charging(self, charging_process, reason):
        """Interrupt an ongoing charging process from within the station."""
        # charging_process: the SimPy Process object returned by env.process(self.charge(...)).
        # .triggered is True once the process has already finished (or been interrupted).
        # We only interrupt if charging is still running.
        if not charging_process.triggered:
            charging_process.interrupt(reason)

class EV:
    def __init__(self, env, name, station):
        self.env = env
        self.name = name
        self.station = station

    def run(self):
        while True:
            # Drive for a while
            yield self.env.timeout(random.randint(20, 40))

            # Park & charge
            print(f't={self.env.now:5.1f} {self.name} start parking')
            # Start the station's charge() as a process -> we get a reference to the charging process
            # that we can later inspect (.triggered) or interrupt.
            charging = self.env.process(self.station.charge(self.name))
            parking = self.env.timeout(random.randint(0, 60))
            yield charging | parking
            # Driver returns: ask the station to stop charging if still in progress
            self.station.abort_charging(charging, 'Need to go!')
            print(f't={self.env.now:5.1f} {self.name} stop parking')

random.seed(42)

env = simpy.Environment()
station = ChargingStation(env, full_charge_time=60)
ev = EV(env, 'EV 1', station)
env.process(ev.run())
env.run(until=100)


t= 40.0 EV 1 start parking
t= 40.0 EV 1 charging started
t= 47.0 EV 1 stop parking
t= 47.0 EV 1 charging interrupted, msg: Need to go!
t= 67.0 EV 1 start parking
t= 67.0 EV 1 charging started


**What's new here?**

* `charging = self.env.process(self.station.charge(self.name))` — we keep a **reference** to the running charging process.
* When the driver comes back, `self.station.abort_charging(charging, 'Need to go!')` calls `charging.interrupt(...)` directly — **no resource is involved**.
* The `charge()` generator wraps its `yield env.timeout(...)` in `try / except simpy.Interrupt` to catch the cancel and log it.


### Event attributes

The `.triggered` check you just saw in `abort_charging` is one of four common event attributes. A SimPy `Process` is itself an `Event`, so these apply to both:

| Attribute | Meaning |
|---|---|
| `triggered` | `True` once `.succeed()` or `.fail()` has been called on the event |
| `processed` | `True` once SimPy has run all callbacks for the event (strictly after `triggered`) |
| `ok` | `True` if fired via `.succeed()`, `False` if via `.fail()` |
| `value` | Return value passed to `.succeed(value=…)` — or the exception passed to `.fail()` |

*(Also present: `env` and `callbacks` — rarely read directly in user code.)*

**Lifecycle:**

```
   created                  succeed() / fail()              callbacks run
────────●───────────────────────────●─────────────────────────────●──────▶
 triggered = False           triggered = True              triggered = True
 processed = False           processed = False             processed = True
```

For a process, `triggered == True` means it has **finished** — so guarding with `if not charging_process.triggered` avoids interrupting a process that's already done.


### ❓ Menti: EV interruption frequency


# Summary

* **Condition events** — wait for multiple events using `|`, `&`, `AnyOf`, `AllOf`
* **Advanced resources:**
    * `PriorityResource` — priority-based access (ER triage)
    * `PreemptiveResource` — high-priority kicks low-priority out
    * `Container` — bulk goods (gas station)
    * `Store` — distinguishable Python objects (mentioned only)
* **Process interruptions** — `process.interrupt()` cancels another process **directly, with no resource involved** (EV charging)
* `PreemptiveResource` vs. `process.interrupt()`: the former happens on a **shared resource** triggered by SimPy; the latter is a **direct** cancel sent by another process.

**Next up:** [exercise2.ipynb](exercise2.ipynb) — apply process interruption + `PreemptiveResource` to model a **machine shop** with breakdowns and a repairman whose other jobs get preempted by repairs.


<img src="images/d3.png" alt="D3 Group logo" style="width:50%; float:center;" />
